# EDA BANKING77

Analise inicial do dataset usado no Fintech Guard.

## Fonte

- Dataset: BANKING77
- Fonte: https://huggingface.co/datasets/PolyAI/banking77
- Repositorio original: https://github.com/PolyAI-LDN/task-specific-datasets
- Licenca: CC BY 4.0

Escolha: o dataset e de atendimento bancario, tem texto de cliente e categoria de intencao, com mais de 500 amostras.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

RAW_DIR = ROOT / 'data' / 'raw' / 'banking77'
PROCESSED_DIR = ROOT / 'data' / 'processed' / 'banking77'
FIGURES_DIR = ROOT / 'reports' / 'figures'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Carregamento

In [ ]:
train_raw = pd.read_csv(RAW_DIR / 'train.csv')
test_raw = pd.read_csv(RAW_DIR / 'test.csv')

pd.DataFrame({
    'split': ['train_raw', 'test_raw'],
    'linhas': [len(train_raw), len(test_raw)],
    'colunas': [train_raw.shape[1], test_raw.shape[1]],
})

## Shape, tipos, ausentes e duplicatas

In [ ]:
def resumo(split, df):
    text_strip = df['text'].astype(str).str.strip()
    return {
        'split': split,
        'shape': df.shape,
        'dtype_text': str(df['text'].dtype),
        'dtype_category': str(df['category'].dtype),
        'missing_text': int(df['text'].isna().sum()),
        'missing_category': int(df['category'].isna().sum()),
        'linhas_duplicadas': int(df.duplicated().sum()),
        'textos_repetidos_apos_strip': int(text_strip.duplicated().sum()),
        'textos_vazios': int(text_strip.eq('').sum()),
        'textos_com_espaco_fora': int(df['text'].astype(str).ne(text_strip).sum()),
        'categorias': int(df['category'].nunique()),
    }

pd.DataFrame([resumo('train_raw', train_raw), resumo('test_raw', test_raw)])

## Distribuicao das categorias

In [ ]:
train_counts = train_raw['category'].value_counts()

pd.concat(
    [train_counts.head(10).rename('mais_frequentes'), train_counts.tail(10).rename('menos_frequentes')],
    axis=1,
)

## Limpeza

In [ ]:
train = train_raw[['text', 'category']].copy()
test = test_raw[['text', 'category']].copy()

train['text'] = train['text'].astype(str).str.strip()
test['text'] = test['text'].astype(str).str.strip()

duplicadas_treino = int(train.duplicated().sum())
duplicadas_teste = int(test.duplicated().sum())

train = train.drop_duplicates().reset_index(drop=True)
test = test.drop_duplicates().reset_index(drop=True)

overlap = train['text'].str.lower().isin(test['text'].str.lower())
overlaps_removidos = int(overlap.sum())
train = train.loc[~overlap].reset_index(drop=True)

train.to_csv(PROCESSED_DIR / 'train.csv', index=False)
test.to_csv(PROCESSED_DIR / 'test.csv', index=False)

pd.DataFrame({
    'item': ['duplicadas_treino', 'duplicadas_teste', 'overlaps_removidos', 'treino_final', 'teste_final'],
    'valor': [duplicadas_treino, duplicadas_teste, overlaps_removidos, len(train), len(test)],
})

Decisoes de limpeza:

- manter apenas `text` e `category`;
- remover espacos no inicio e no fim;
- remover duplicatas exatas depois do `strip()`;
- remover do treino mensagens que tambem aparecem no teste;
- manter caixa e pontuacao, porque isso pode ajudar numa etapa de modelo.

## Tamanho das mensagens

In [ ]:
pd.DataFrame({
    'train': train['text'].str.len().describe(),
    'test': test['text'].str.len().describe(),
}).round(2)

## Graficos

In [ ]:
counts = train['category'].value_counts().sort_values(ascending=True)

plt.figure(figsize=(10, 18))
plt.barh(counts.index, counts.values, color='#1f77b4')
plt.title('Distribuicao das categorias - treino')
plt.xlabel('Quantidade')
plt.ylabel('Categoria')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'banking77_category_distribution.png', dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(train['text'].str.len(), bins=35, alpha=0.75, label='treino', color='#1f77b4')
plt.hist(test['text'].str.len(), bins=35, alpha=0.55, label='teste', color='#ff7f0e')
plt.title('Tamanho das mensagens')
plt.xlabel('Caracteres')
plt.ylabel('Quantidade')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'banking77_text_length_distribution.png', dpi=160)
plt.show()

## Hipoteses

1. Cartao deve ser um tema forte no atendimento: chegada, bloqueio, falha, troca, pagamento nao reconhecido e cartao virtual aparecem em varias categorias.
2. Transferencia e pagamento devem gerar bastante demanda, porque o dataset separa casos de taxa, pendencia, falha, cancelamento e destinatario que nao recebeu.
3. Ha um bloco de intencoes ligadas a seguranca da conta: identidade, origem dos fundos, aparelho perdido, cartao comprometido e PIN.